In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# Patent embeddings, matching and CPC projection

Prepare PaECTER and PatEmbed matches for SI S3.5.1–3 and additional FSE matches for the larger-match settings in SI S3.5.4. Run from `revision_notebooks/` with paths set in the first cell. Shared text, concordance and projection functions are in `embedding_utils.py`.

The four sections cover encoding, family matching, CPC projection and extension of FSE matches. Run the imports cell before selecting a section. Encoding uses local model snapshots and CUDA by default.


In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from embedding_utils import (
    PAIR_COLUMNS, PROJECTION_COLUMNS, build_texts, build_universe,
    ordinary_cpc, select_pairs, write_projection, append_candidates,
)

MODELS = ['paecter', 'patembed_base']
EMBEDDINGS = DATA / 'embeddings'
EMBEDDINGS.mkdir(parents=True, exist_ok=True)
TEXT_COLUMNS = ['patent_id', 'patent_title', 'patent_abstract']
DEVICE = 'cuda'
BATCH_SIZE = 256
THREADS = 20

## 1. Encode publication title and abstract

Read `patent_id,patent_title,patent_abstract` from `PATENT_TEXT/g_patent_title_abstract.parquet`. Strip title and abstract and join them with two newlines. Missing fields become empty strings; every publication keeps a concordance row.

Place these SentenceTransformer snapshots under `DATA/models/`:

| Folder | Model and commit |
|---|---|
| `paecter` | `mpi-inno-comp/paecter`, `1f355d9e9a34dc4859e22d05ce27e970eeacfbd2` |
| `patembed_base` | `datalyes/patembed-base`, `244ba452908a3093be11efb8a44396652317fb14` |

Use maximum sequence length 512, batch size 256 and normalized float32 outputs. Encode PatEmbed without a prompt. Sort each 50,000-row chunk by character length for inference, then restore input order. PaECTER has 1,024 dimensions and PatEmbed 768.

FSE/PARANMT-300 encoding is in [dataprep 01](../dataprep/01_fse_embeddings_and_scores.ipynb).


In [ ]:
from sentence_transformers import SentenceTransformer

text_file = pq.ParquetFile(PATENT_TEXT / 'g_patent_title_abstract.parquet')
for model_key in MODELS:
    model = SentenceTransformer(str(DATA / 'models' / model_key), device=DEVICE)
    model.max_seq_length = 512
    model.default_prompt_name = None
    output = EMBEDDINGS / model_key
    output.mkdir(parents=True, exist_ok=True)
    embeddings = np.lib.format.open_memmap(
        output / 'embeddings.npy', mode='w+', dtype=np.float32,
        shape=(text_file.metadata.num_rows, model.get_sentence_embedding_dimension()),
    )
    row_number = 0
    writer = None
    try:
        for batch in text_file.iter_batches(batch_size=50_000, columns=TEXT_COLUMNS):
            texts = build_texts(batch.to_pandas())
            order = np.argsort([-len(text) for text in texts])
            encoded = model.encode(
                [texts[i] for i in order], batch_size=BATCH_SIZE, prompt='',
                convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True,
            ).astype(np.float32, copy=False)
            restored = np.empty_like(encoded)
            restored[order] = encoded
            end = row_number + len(texts)
            embeddings[row_number:end] = restored
            concordance = pa.table({
                'row_number': np.arange(row_number, end, dtype=np.int64),
                'patent_id': batch.column('patent_id'),
            })
            if writer is None:
                writer = pq.ParquetWriter(output / 'concordance.parquet', concordance.schema)
            writer.write_table(concordance)
            row_number = end
            print(f'{model_key}: encoded {end:,}/{text_file.metadata.num_rows:,}', flush=True)
    finally:
        if writer is not None:
            writer.close()
    embeddings.flush()
    del model, embeddings

## 2. Match patent families with alternative vectors

Read `BASELINE/patvecdf.parquet`, saved by `../0_get_familiy_id.ipynb`. It maps publication `patent_id` to `docdb_family_id` in FSE array order. Remove rows with missing IDs and select the first publication per family. A Y02/Y04 assignment identifies a green family; the remaining families are matching candidates.

For each alternative embedding, join the selected publication ID to its `row_number` concordance to locate the vector.


In [ ]:
patvec = pd.read_parquet(BASELINE / 'patvecdf.parquet', columns=['patent_id', 'docdb_family_id'])
green_assignments = pd.read_parquet(PATSTAT / 'tls225.parquet', columns=['docdb_family_id', 'cpc4d'])
universe = build_universe(patvec, green_assignments)
universe.to_parquet(EMBEDDINGS / 'universe.parquet', index=False)
universe.groupby('ccmt').size()

Use HNSW cosine distance with `M=32`, `efConstruction=200`, `post=2`, `efSearch=200` and 20 threads. Query 50 neighbours and save the first ten before applying the matching filters.

The saved table contains `ccmt_patent_id,patent_id,cos,rank`. Both patent-ID columns hold family IDs; `cos` is cosine distance.


In [ ]:
import nmslib

for model_key in MODELS:
    output = EMBEDDINGS / model_key
    embeddings = np.load(output / 'embeddings.npy', mmap_mode='r')
    concordance = pd.read_parquet(output / 'concordance.parquet', columns=['row_number', 'patent_id'])
    model_universe = universe.drop(columns='embedding_row').merge(
        concordance.rename(columns={'row_number': 'embedding_row'}),
        on='patent_id', how='left', sort=False,
    )
    model_universe['embedding_row'] = model_universe['embedding_row'].astype(np.int64)
    focal = model_universe.loc[model_universe['ccmt']]
    other = model_universe.loc[~model_universe['ccmt']]
    other_ids = other['docdb_family_id'].to_numpy()

    index = nmslib.init(method='hnsw', space='cosinesimil')
    index.addDataPointBatch(np.asarray(embeddings[other['embedding_row']], dtype=np.float32))
    index.createIndex(
        {'M': 32, 'efConstruction': 200, 'post': 2, 'indexThreadQty': THREADS},
        print_progress=True,
    )
    index.setQueryTimeParams({'efSearch': 200})
    writer = None
    try:
        for start in range(0, len(focal), 10_000):
            batch = focal.iloc[start:start + 10_000]
            results = index.knnQueryBatch(
                np.asarray(embeddings[batch['embedding_row']], dtype=np.float32),
                k=50, num_threads=THREADS,
            )
            rows = [
                (family, other_ids[int(candidate)], float(distance), rank)
                for family, (candidates, distances) in zip(batch['docdb_family_id'], results)
                for rank, (candidate, distance) in enumerate(zip(candidates[:10], distances[:10]), start=1)
            ]
            table = pa.Table.from_pandas(pd.DataFrame(rows, columns=PAIR_COLUMNS), preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(output / 'matched_top10.parquet', table.schema)
            writer.write_table(table)
            print(f'{model_key}: matched {start + len(batch):,}/{len(focal):,}', flush=True)
    finally:
        if writer is not None:
            writer.close()
    del index, embeddings

## 3. Detailed CPC projection for each alternative

Read each model's `matched_top10.parquet`, family CPC assignments from `PATSTAT/tls225.parquet`, and the 50-dimensional Word2Vec model `BASELINE/external/cpc_sg_ns_dim50.model`. [Dataprep 02](../dataprep/02_cpc_corpus_preparation.ipynb) trains this CPC2vec model.

Select top-ten ranks, then retain distances in `[0.01, 0.5]`. Remove spaces from CPC symbols and retain each family/code combination once, using ordinary codes in the model vocabulary. Require codes for both families and at least one shared four-character CPC subclass. Keep the retrieval ranks of surviving pairs.

For every focal-family code, `leastsimilar` is the maximum normalized CPC cosine against codes of the matched family. The distance score is `1 - leastsimilar`; text-similarity strength is `1 - cos`. GBB extraction selects codes whose distance score exceeds the pair's third quartile by more than 1.5 interquartile ranges.


In [ ]:
from gensim.models import Word2Vec

vectors = Word2Vec.load(str(BASELINE / 'external' / 'cpc_sg_ns_dim50.model')).wv
vectors.unit_normalize_all()
assignments = pd.read_parquet(PATSTAT / 'tls225.parquet', columns=['docdb_family_id', 'cpc_class_symbol'])
for model_key in MODELS:
    output = EMBEDDINGS / model_key
    pairs = pd.read_parquet(output / 'matched_top10.parquet')
    cpc = ordinary_cpc(assignments, pairs, vectors)
    filtered = select_pairs(pairs, cpc, maximum_rank=10)
    filtered.to_parquet(output / 'matched_ccmt_other_filtered.parquet', index=False)
    write_projection(filtered, cpc, vectors, output / 'green_proj_cosangle.parquet')
    print(model_key, 'retained pairs:', len(filtered), 'focal families:', filtered.ccmt_patent_id.nunique())

The filtered match table adds `samecpc4d`, the number of shared CPC subclasses. Projection rows contain `ccmt_patent_id,patent_id,cpc,leastsimilar,cos,rank`. Save each model's matches and projections under `DATA/embeddings/` for notebooks 05 and 06.


## 4. Extend FSE matches to ranks 11–20

Read the following files:

- `BASELINE/matched_ccmt_other.parquet`: unfiltered top-ten family matches from `../1_ccmt_matching.ipynb`.
- `BASELINE/green_proj_cosangle.parquet`: detailed CPC projection from `../2_cpc.ipynb`.
- `BASELINE/patvecdf.parquet` and `PATSTAT/tls225.parquet`: family concordance and CPC assignments.
- `PATENT_TEXT/fse_embedding/title_abstract_embedding.npy`: FSE publication vectors.
- `BASELINE/hnsw_cos_index_patvec_other.bin` and its saved-data companion: the nongreen index from `../1_ccmt_matching.ipynb`.
- The detailed CPC2vec model used in section 3.

Keep the saved top ten and their projection values. Query the saved index for 50 candidates, select ten additional matches, and filter/project those appended pairs. Index positions follow the nongreen family order established by the concordance.


In [ ]:
import nmslib

fse_output = EMBEDDINGS / 'fse'
fse_output.mkdir(parents=True, exist_ok=True)
submitted = pd.read_parquet(BASELINE / 'matched_ccmt_other.parquet', columns=PAIR_COLUMNS[:3])
submitted = submitted.sort_values(['ccmt_patent_id', 'cos', 'patent_id']).reset_index(drop=True)
submitted['rank'] = submitted.groupby('ccmt_patent_id').cumcount() + 1
submitted = submitted.loc[submitted['rank'].le(10)]

patvec = pd.read_parquet(BASELINE / 'patvecdf.parquet', columns=['patent_id', 'docdb_family_id'])
green_assignments = pd.read_parquet(PATSTAT / 'tls225.parquet', columns=['docdb_family_id', 'cpc4d'])
fse_universe = build_universe(patvec, green_assignments)
fse_vectors = np.load(PATENT_TEXT / 'fse_embedding' / 'title_abstract_embedding.npy', mmap_mode='r')
# Match the vector population in root ../1_ccmt_matching.ipynb.
valid = np.zeros(len(fse_universe), dtype=bool)
for start in range(0, len(fse_universe), 50_000):
    values = fse_vectors[fse_universe.embedding_row.iloc[start:start + 50_000]]
    valid[start:start + len(values)] = np.isfinite(values).all(axis=1) & (np.linalg.norm(values, axis=1) > 0)
fse_universe = fse_universe.loc[valid]
focal = fse_universe.loc[fse_universe['ccmt'] & fse_universe['docdb_family_id'].isin(submitted['ccmt_patent_id'])]
other_ids = fse_universe.loc[~fse_universe['ccmt'], 'docdb_family_id'].to_numpy()
fse_index = nmslib.init(method='hnsw', space='cosinesimil')
fse_index.loadIndex(str(BASELINE / 'hnsw_cos_index_patvec_other.bin'), load_data=True)

In [ ]:
writer = None
try:
    for start in range(0, len(focal), 5_000):
        batch = focal.iloc[start:start + 5_000]
        results = fse_index.knnQueryBatch(
            np.asarray(fse_vectors[batch['embedding_row']], dtype=np.float32),
            k=50, num_threads=35,
        )
        old = submitted.loc[submitted['ccmt_patent_id'].isin(batch['docdb_family_id'])]
        extended = append_candidates(old, other_ids, dict(zip(batch['docdb_family_id'], results)))
        table = pa.Table.from_pandas(extended, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(fse_output / 'matched_top20.parquet', table.schema)
        writer.write_table(table)
        print(f'Extended FSE {start + len(batch):,}/{len(focal):,} focal families', flush=True)
finally:
    if writer is not None:
        writer.close()
del fse_index, fse_vectors

In [ ]:
from gensim.models import Word2Vec

extended = pd.read_parquet(fse_output / 'matched_top20.parquet')
extension = extended.loc[extended['rank'].gt(10)].copy()
vectors = Word2Vec.load(str(BASELINE / 'external' / 'cpc_sg_ns_dim50.model')).wv
vectors.unit_normalize_all()
assignments = pd.read_parquet(PATSTAT / 'tls225.parquet', columns=['docdb_family_id', 'cpc_class_symbol'])
cpc = ordinary_cpc(assignments, extension, vectors)
filtered = select_pairs(extension, cpc, maximum_rank=20)
filtered.to_parquet(fse_output / 'matched_ranks11_20_filtered.parquet', index=False)
extension_path = fse_output / 'green_proj_cosangle_ranks11_20.parquet'
write_projection(filtered, cpc, vectors, extension_path)

In [ ]:
original = pd.read_parquet(BASELINE / 'green_proj_cosangle.parquet', columns=PROJECTION_COLUMNS[:-1])
original = original.merge(
    submitted[['ccmt_patent_id', 'patent_id', 'rank']],
    on=['ccmt_patent_id', 'patent_id'], how='left', sort=False,
)
combined = pd.concat([original, pd.read_parquet(extension_path)], ignore_index=True)
combined[PROJECTION_COLUMNS].to_parquet(fse_output / 'green_proj_cosangle_top20.parquet', index=False)
combined.groupby('rank').size()

Notebook 06 reads `DATA/embeddings/fse/green_proj_cosangle_top20.parquet`: ranks 1–10 contain the saved FSE projection, and ranks 11–20 contain the additional pairs. Notebook 05 reads the PaECTER and PatEmbed match tables and projections from sections 2–3.
